# Análises de Negócio — Marketing e Atendentes


4. **Diretora de Marketing** — satisfação era 85% no início de 2025, piorou, e ela atribui a piora a fatores externos
5. **Atendentes (relato)** — alta demanda por causa de app/site lentos e de difícil navegação; URA com caminhos confusos

**Convenções:** `FK_status_chamado` 1 = Resolvido, 2 = Não resolvido, 3 = Abandonado; `SK = -1` = "Não Informado".


In [0]:
%sql
USE CATALOG vcredit_medallion;

## 4. Diretora de Marketing — Satisfação do Cliente

> "No começo de 2025 nossos clientes nos amavam, o coeficiente de satisfação da V-Credit era 85%. Sentimos uma piora recente, mas acredito que esteja ligada a fatores externos e não à nossa operação. Não deve estar muito diferente do que vimos no ano passado."

**Afirmações a testar com dados:**
- (a) A satisfação era ~85% no começo de 2025
- (b) Houve piora recente
- (c) A piora é externa, não operacional
- (d) "Não deve estar muito diferente do ano passado"

**Lógica do teste:** fator externo (economia, concorrência, sazonalidade) deveria derrubar a nota de forma **uniforme** em todos os canais, motivos e níveis, e **também no mesmo mês do ano anterior**. Se a queda se concentra em canais/níveis específicos ou acompanha espera, abandono e não resolução, a causa é operacional.

**Análises:**
- 4.1 Evolução mensal do CSAT (duas definições)
- 4.2 Comparação ano contra ano, mesmos meses (testa a afirmação "não é diferente do ano passado")
- 4.3 CSAT por status do chamado (resolvido x não resolvido)
- 4.4 Queda por canal: uniforme ou concentrada?
- 4.5 Queda por categoria e criticidade do motivo
- 4.6 CSAT por faixa de tempo de espera, 2025 x 2026
- 4.7 Viés de resposta: quem não responde a pesquisa?
- 4.8 CSAT por perfil de cliente (idade e região)

In [0]:
%sql
-- 4.1 — Evolução mensal do CSAT, em duas definições
-- csat_pct_4_5: % de notas 4 ou 5 entre os chamados avaliados
-- csat_nota_media_pct: nota média / 5 (escala 0–100)
SELECT
  d.ano,
  d.mes,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                         AS avaliacoes,
  ROUND(AVG(f.nota_avaliacao), 2)                                AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5,
  ROUND(100 * AVG(f.nota_avaliacao) / 5, 1)                      AS csat_nota_media_pct
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

In [0]:
%sql
-- 4.2 — Ano contra ano, mesmos meses (jan–jul de 2025 x 2026)
-- Dados vão até jul/2026, então só jan–jul permite comparar nas mesmas condições
SELECT
  d.mes,
  ROUND(100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1)    AS variacao_pp,
  COUNT_IF(d.ano = 2025)                                                              AS chamados_2025,
  COUNT_IF(d.ano = 2026)                                                              AS chamados_2026
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1 AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY d.mes
ORDER BY d.mes;

In [0]:
%sql
-- 4.3 — CSAT por status do chamado
-- Se a nota depende de o problema ter sido resolvido, a satisfação é função da operação
SELECT
  d.ano,
  CASE f.FK_status_chamado
    WHEN 1 THEN 'Resolvido'
    WHEN 2 THEN 'Não resolvido (atendido)'
    WHEN 3 THEN 'Abandonado (na espera)'
  END                                                              AS status_chamado,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano), 1) AS pct_do_ano,
  COUNT_IF(f.nota_avaliacao IS NOT NULL)                           AS avaliacoes,
  ROUND(AVG(f.nota_avaliacao), 2)                                  AS nota_media
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, f.FK_status_chamado
ORDER BY d.ano, f.FK_status_chamado;

In [0]:
%sql
-- 4.4 — A queda é uniforme (externa) ou concentrada (operacional)?
-- Compara jan–jul/2025 x jan–jul/2026 por canal: CSAT, variação e mudança de peso no volume
WITH base AS (
  SELECT
    c.nome AS canal, c.tipo AS tipo_canal, c.nivel_etapa, d.ano,
    f.nota_avaliacao
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1
    AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
)
SELECT
  canal, tipo_canal, nivel_etapa,
  ROUND(100 * COUNT_IF(ano = 2025) / SUM(COUNT_IF(ano = 2025)) OVER (), 1) AS pct_volume_2025,
  ROUND(100 * COUNT_IF(ano = 2026) / SUM(COUNT_IF(ano = 2026)) OVER (), 1) AS pct_volume_2026,
  ROUND(100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS variacao_pp
FROM base
GROUP BY canal, tipo_canal, nivel_etapa
ORDER BY variacao_pp ASC;

In [0]:
%sql
-- 4.5 — Queda por categoria e criticidade do motivo (jan–jul/2025 x jan–jul/2026)
-- Queda concentrada em categorias específicas aponta para falha de processo/produto, não para macroeconomia
WITH base AS (
  SELECT m.categoria, m.criticidade, d.ano, f.nota_avaliacao
  FROM gold.fato_chamado f
  JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
  WHERE d.SK_data <> -1 AND m.SK_motivo <> -1
    AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
)
SELECT
  categoria, criticidade,
  COUNT_IF(ano = 2026)                                                           AS chamados_2026,
  ROUND(100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026,
  ROUND(
      100 * COUNT_IF(ano = 2026 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2026 AND nota_avaliacao IS NOT NULL), 0)
    - 100 * COUNT_IF(ano = 2025 AND nota_avaliacao >= 4)
          / NULLIF(COUNT_IF(ano = 2025 AND nota_avaliacao IS NOT NULL), 0), 1) AS variacao_pp
FROM base
GROUP BY categoria, criticidade
ORDER BY variacao_pp ASC;

In [0]:
%sql
-- 4.6 — CSAT por faixa de tempo de espera, 2025 x 2026
-- Dois efeitos: (1) nota cai quando a espera aumenta; (2) mais chamados caem nas faixas longas em 2026
SELECT
  d.ano,
  CASE
    WHEN f.tempo_espera = 0   THEN '0 min (sem espera)'
    WHEN f.tempo_espera <= 60  THEN 'até 1 min'
    WHEN f.tempo_espera <= 180 THEN '1 a 3 min'
    WHEN f.tempo_espera <= 300 THEN '3 a 5 min'
    WHEN f.tempo_espera <= 600 THEN '5 a 10 min'
    ELSE 'mais de 10 min'
  END                                                              AS faixa_espera,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano), 1) AS pct_do_ano,
  ROUND(AVG(f.nota_avaliacao), 2)                                  AS nota_media,
  ROUND(100 * COUNT_IF(f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_pct_4_5
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.tipo = 'Humano' AND d.ano IN (2025, 2026)
GROUP BY d.ano, 2
ORDER BY d.ano,
  CASE faixa_espera
    WHEN '0 min (sem espera)' THEN 1
    WHEN 'até 1 min'          THEN 2
    WHEN '1 a 3 min'          THEN 3
    WHEN '3 a 5 min'          THEN 4
    WHEN '5 a 10 min'         THEN 5
    ELSE 6
  END;

In [0]:
%sql
-- 4.7 — Viés de resposta: quem responde a pesquisa?
-- Cliente que abandona normalmente não avalia. Se o abandono cresce, o CSAT medido
-- pode estar SUPERESTIMADO: a piora real seria maior do que a que o Marketing enxerga
SELECT
  d.ano,
  CASE f.FK_status_chamado
    WHEN 1 THEN 'Resolvido'
    WHEN 2 THEN 'Não resolvido (atendido)'
    WHEN 3 THEN 'Abandonado (na espera)'
  END                                                              AS status_chamado,
  COUNT(*)                                                         AS chamados,
  ROUND(100 * COUNT_IF(f.nota_avaliacao IS NOT NULL) / COUNT(*), 1) AS taxa_resposta_pct
FROM gold.fato_chamado f
JOIN gold.dim_data d ON f.FK_data_abertura = d.SK_data
WHERE d.SK_data <> -1
GROUP BY d.ano, f.FK_status_chamado
ORDER BY d.ano, f.FK_status_chamado;

In [0]:
%sql
-- 4.8 — CSAT por perfil de cliente (faixa etária e região), 2025 x 2026
-- Se a queda atinge todos os perfis de forma parecida, o problema é do serviço e não de um público específico
-- ATENÇÃO: assume gold.dim_cliente.regiao; ajustar o nome da coluna se for diferente
SELECT
  cl.faixa_idade,
  cl.regiao,
  COUNT_IF(d.ano = 2026)                                                           AS chamados_2026,
  ROUND(100 * COUNT_IF(d.ano = 2025 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2025 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2025,
  ROUND(100 * COUNT_IF(d.ano = 2026 AND f.nota_avaliacao >= 4)
            / NULLIF(COUNT_IF(d.ano = 2026 AND f.nota_avaliacao IS NOT NULL), 0), 1) AS csat_2026
FROM gold.fato_chamado f
JOIN gold.dim_data    d  ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_cliente cl ON f.FK_cliente = cl.SK_cliente
WHERE d.SK_data <> -1 AND cl.SK_cliente <> -1
  AND cl.faixa_idade <> 'Não Informado'
  AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY cl.faixa_idade, cl.regiao
HAVING COUNT_IF(d.ano = 2026) > 30
ORDER BY (csat_2026 - csat_2025) ASC
LIMIT 20;

## 5. Atendentes (relato) — Qualidade do App/Site e da URA

> "A alta demanda deve-se à qualidade do app e do site, que são lentos e possuem navegação difícil. Já ouvimos queixas dos usuários sobre a URA possuir caminhos confusos e de difícil navegação."

**Limitação importante:** o banco não tem telemetria de app/site (tempo de carregamento, cliques, erros). Então não dá para medir "lentidão" diretamente. Testamos o relato por **proxies indiretos** presentes nos chamados:
- chamados cujo motivo é problema técnico nos canais digitais
- demandas simples e "autoserviçáveis" que mesmo assim chegam ao atendimento humano (falha da etapa Prevenção/Autosserviço do funil)
- sinais de URA confusa: abandono, resolução baixa, cliente que desiste da URA e liga de novo ou cai no humano
- recontato (cliente volta pelo mesmo motivo em poucos dias)

**Análises:**
- 5.1 Peso dos problemas técnicos em canais digitais ao longo do tempo
- 5.2 Crescimento do volume por canal
- 5.3 Funil real x funil desejado
- 5.4 Motivos simples que chegam ao humano
- 5.5 URA: resolução, abandono e "escape" para o humano
- 5.6 Recontato por canal (retrabalho)
- 5.7 Dimensionamento do ganho (premissas explícitas)

In [0]:
%sql
-- 5.1 — Peso dos problemas técnicos nos canais digitais (categoria 'Atendimento') ao longo do tempo
-- Categoria 'Atendimento' = problemas técnicos em site, app, chatbot e URA (conforme dicionário do case)
SELECT
  d.ano,
  d.mes,
  COUNT(*)                                                           AS chamados,
  COUNT_IF(m.categoria = 'Atendimento')                              AS chamados_problema_tecnico,
  ROUND(100 * COUNT_IF(m.categoria = 'Atendimento') / COUNT(*), 1)   AS pct_problema_tecnico,
  COUNT_IF(m.nome LIKE '%plicativo%')                                AS chamados_problema_app
FROM gold.fato_chamado f
JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
WHERE d.SK_data <> -1 AND m.SK_motivo <> -1
GROUP BY d.ano, d.mes
ORDER BY d.ano, d.mes;

In [0]:
%sql
-- 5.2 — Crescimento do volume por canal (jan–jul/2025 x jan–jul/2026)
-- A demanda cresceu onde? Em autosserviço (URA/Chatbot) ou nos canais humanos?
SELECT
  c.nome                                                   AS canal,
  c.tipo                                                   AS tipo_canal,
  COUNT_IF(d.ano = 2025)                                   AS chamados_2025,
  COUNT_IF(d.ano = 2026)                                   AS chamados_2026,
  ROUND(100 * (COUNT_IF(d.ano = 2026) - COUNT_IF(d.ano = 2025))
            / NULLIF(COUNT_IF(d.ano = 2025), 0), 1)        AS crescimento_pct
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.SK_canal <> -1
  AND d.ano IN (2025, 2026) AND d.mes BETWEEN 1 AND 7
GROUP BY c.nome, c.tipo, c.nivel_etapa
ORDER BY c.nivel_etapa;

In [0]:
%sql
-- 5.3 — Funil real x funil desejado
-- O case quer menos volume nos níveis humanos e mais em Prevenção/Autosserviço.
-- Aqui medimos quanto do volume registrado está em autosserviço x humano, por semestre
SELECT
  d.ano,
  CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END AS semestre,
  c.tipo                                                         AS tipo_canal,
  c.nome                                                         AS canal,
  COUNT(*)                                                       AS chamados,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY d.ano, CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END), 1) AS pct_do_semestre,
  ROUND(AVG(f.custo), 2)                                         AS custo_medio
FROM gold.fato_chamado f
JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
WHERE d.SK_data <> -1 AND c.SK_canal <> -1
GROUP BY d.ano, CASE WHEN d.mes <= 6 THEN '1º semestre' ELSE '2º semestre' END, c.tipo, c.nome, c.nivel_etapa
ORDER BY d.ano, 2, c.nivel_etapa;

In [0]:
%sql
-- 5.4 — Motivos simples que chegam ao atendimento humano
-- Consultas de fatura, limite, contrato, pontos e alterações cadastrais deveriam resolver em Prevenção (site/app).
-- Cada uma que chega ao humano é sinal de que o app/site não está cumprindo o papel
SELECT
  m.nome                                                              AS motivo,
  m.categoria,
  COUNT(*)                                                            AS chamados_total,
  COUNT_IF(c.tipo = 'Humano')                                         AS chamados_humano,
  ROUND(100 * COUNT_IF(c.tipo = 'Humano') / COUNT(*), 1)              AS pct_no_humano,
  ROUND(SUM(CASE WHEN c.tipo = 'Humano' THEN f.custo ELSE 0 END), 2)  AS custo_humano,
  ROUND(AVG(CASE WHEN c.tipo = 'Humano' THEN f.custo END), 2)         AS custo_medio_humano,
  ROUND(AVG(CASE WHEN c.tipo <> 'Humano' THEN f.custo END), 2)        AS custo_medio_autosservico
FROM gold.fato_chamado f
JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
JOIN gold.dim_canal  c ON f.FK_canal  = c.SK_canal
WHERE m.SK_motivo <> -1 AND c.SK_canal <> -1
  AND (m.nome LIKE 'Consulta%' OR m.nome LIKE '%ontos%' OR m.categoria = 'Cadastral')
GROUP BY m.nome, m.categoria
ORDER BY custo_humano DESC;

In [0]:
%sql
-- 5.5 — URA: resolução, abandono e "escape" para o humano
-- Sinal de URA confusa: cliente desiste/não resolve na URA e abre novo chamado em até 1 dia em canal humano
-- ATENÇÃO: assume gold.dim_data.data (DATE). Granularidade diária: o "escape" mede até 1 dia depois
WITH seq AS (
  SELECT
    f.FK_cliente,
    c.nome AS canal,
    c.tipo AS tipo_canal,
    f.FK_status_chamado AS status,
    f.tempo_atendimento,
    d.data AS dt,
    LEAD(c.tipo) OVER (PARTITION BY f.FK_cliente ORDER BY d.data, c.nivel_etapa) AS prox_tipo,
    LEAD(d.data) OVER (PARTITION BY f.FK_cliente ORDER BY d.data, c.nivel_etapa) AS prox_dt
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1 AND f.FK_cliente <> -1
)
SELECT
  canal,
  COUNT(*)                                                           AS chamados,
  ROUND(100 * COUNT_IF(status = 1) / COUNT(*), 1)                    AS pct_resolucao,
  ROUND(100 * COUNT_IF(status = 3) / COUNT(*), 1)                    AS pct_abandono,
  ROUND(AVG(tempo_atendimento), 1)                                   AS tempo_medio_seg,
  ROUND(100 * COUNT_IF(prox_tipo = 'Humano' AND DATEDIFF(prox_dt, dt) <= 1) / COUNT(*), 1)
                                                                     AS pct_escape_para_humano_1d,
  ROUND(100 * COUNT_IF(status <> 1 AND prox_tipo = 'Humano' AND DATEDIFF(prox_dt, dt) <= 1)
            / NULLIF(COUNT_IF(status <> 1), 0), 1)                   AS pct_escape_quando_nao_resolveu
FROM seq
WHERE tipo_canal <> 'Humano'
GROUP BY canal
ORDER BY pct_escape_para_humano_1d DESC;

In [0]:
%sql
-- 5.6 — Recontato: cliente volta pelo mesmo motivo em até 7 dias (retrabalho)
-- Proxy de "a primeira tentativa não resolveu", seja por navegação confusa ou por atendimento ineficaz
-- ATENÇÃO: assume gold.dim_data.data (DATE)
WITH seq AS (
  SELECT
    c.nome AS canal,
    f.custo,
    d.data AS dt,
    LEAD(d.data)       OVER (PARTITION BY f.FK_cliente, f.FK_motivo ORDER BY d.data) AS prox_dt
  FROM gold.fato_chamado f
  JOIN gold.dim_data  d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_canal c ON f.FK_canal = c.SK_canal
  WHERE d.SK_data <> -1 AND c.SK_canal <> -1
    AND f.FK_cliente <> -1 AND f.FK_motivo <> -1
)
SELECT
  canal,
  COUNT(*)                                                         AS chamados,
  COUNT_IF(DATEDIFF(prox_dt, dt) <= 7)                             AS chamados_com_recontato,
  ROUND(100 * COUNT_IF(DATEDIFF(prox_dt, dt) <= 7) / COUNT(*), 1)  AS pct_recontato_7d,
  ROUND(SUM(CASE WHEN DATEDIFF(prox_dt, dt) <= 7 THEN custo ELSE 0 END), 2) AS custo_retrabalho
FROM seq
GROUP BY canal
ORDER BY pct_recontato_7d DESC;

In [0]:
%sql
-- 5.7 — Dimensionamento do ganho (PREMISSAS EXPLÍCITAS — ajustar conforme validação com a V-Credit)
-- Proposta: melhorar app/site e URA para que parte dos motivos simples seja resolvida em autosserviço
-- Ganho anual estimado = chamados simples no humano (anualizado) x taxa de migração x (custo humano − custo autosserviço)
WITH params AS (
  SELECT 0.30 AS taxa_migracao          -- premissa: 30% desses chamados passam a resolver em autosserviço
),
simples AS (
  SELECT
    COUNT_IF(c.tipo = 'Humano')                      AS chamados_humano,
    AVG(CASE WHEN c.tipo = 'Humano'  THEN f.custo END) AS custo_medio_humano,
    AVG(CASE WHEN c.tipo <> 'Humano' THEN f.custo END) AS custo_medio_autosservico,
    COUNT(DISTINCT CONCAT(d.ano, '-', d.mes))        AS meses_na_base
  FROM gold.fato_chamado f
  JOIN gold.dim_data   d ON f.FK_data_abertura = d.SK_data
  JOIN gold.dim_motivo m ON f.FK_motivo = m.SK_motivo
  JOIN gold.dim_canal  c ON f.FK_canal  = c.SK_canal
  WHERE d.SK_data <> -1 AND m.SK_motivo <> -1 AND c.SK_canal <> -1
    AND (m.nome LIKE 'Consulta%' OR m.nome LIKE '%ontos%' OR m.categoria = 'Cadastral')
)
SELECT
  s.chamados_humano,
  ROUND(s.custo_medio_humano, 2)       AS custo_medio_humano,
  ROUND(s.custo_medio_autosservico, 2) AS custo_medio_autosservico,
  p.taxa_migracao,
  ROUND(s.chamados_humano * p.taxa_migracao
        * (s.custo_medio_humano - s.custo_medio_autosservico), 2)                      AS economia_no_periodo_rs,
  ROUND(s.chamados_humano * p.taxa_migracao
        * (s.custo_medio_humano - s.custo_medio_autosservico) / s.meses_na_base * 12, 2) AS economia_anualizada_rs
FROM simples s CROSS JOIN params p;